In [1]:
import torch
import time
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scanpy as sc
import copy
from torch.utils.data import IterableDataset
from torch.utils.data import DataLoader


print("torch version:", torch.__version__)
print("cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))

torch version: 2.5.1
cuda version: 12.1
cuda available: True
gpu count: 1
gpu name: NVIDIA GeForce RTX 3070


In [ ]:
# test
device = torch.device("cuda")
x = torch.randn(10000,10000,device=device)
y = torch.randn(10000, 10000, device=device)

# 让 GPU 忙起来
for i in range(10):
    z = x @ y
    torch.cuda.synchronize()
    print(f"Iteration {i}")
    time.sleep(0.5)

In [ ]:
device = "cuda"
lstm = nn.LSTM(
    input_size=128,
    hidden_size=256,
    num_layers=2,
    batch_first=True
).to(device)

x = torch.randn(64, 100, 128).to(device)

out, _ = lstm(x)

print(out.device)

In [ ]:
# a lstm example
batch_size = 16
seq_len = 10
input_size = 1
hidden_size = 32

x = torch.randn(batch_size,seq_len,input_size)
y = x[:,-1,:].sum(dim=1)

class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size = input_size,
            hidden_size = hidden_size,
            batch_first = True # the order of input tensors (B,T,I)
        )
        self.fc = nn.Linear(hidden_size,1) # fully connected layer

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (h_n, c_n) = self.lstm(x)
        # out: (batch, seq_len, hidden_size)

        last_out = out[:, -1, :]      # 取最后一个时间步
        y_pred = self.fc(last_out)    # (batch, 1)

        return y_pred.squeeze(-1)     # (batch,)


model = SimpleLSTM()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)   
loss_records = []
for epoch in range(40):
    optimizer.zero_grad()      # 1️⃣ 清空梯度

    y_pred = model(x)          # 2️⃣ 前向传播
    loss = criterion(y_pred, y)# 3️⃣ 计算 loss

    loss.backward()            # 4️⃣ 反向传播
    optimizer.step()           # 5️⃣ 更新参数

    #print(f"Epoch {epoch}, loss = {loss.item():.4f}")
    loss_records = loss_records + [loss.item()]

plt.plot(range(len(loss_records)),loss_records)


In [ ]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, promoter_file, scrna_file):
        self.promoter = pd.read_csv(promoter_file)   
        self.scrna = sc.read(scrna_file,sparse=True) 
        #self.y = y_array            

        self.P = self.promoter.shape[0]
        self.C = self.scrna.shape[0]

    def __len__(self):
        return self.P * self.C

    def __getitem__(self, idx):
        promoter_id = idx // self.C
        cell_id = idx % self.C

        promoter
        gene_id = self.promoter[promoter_id]['gene_id'])

        promoter = self.promoter[promoter_id]['sequence']        # (400, 5)
        expr = self.scrna[cell_id]   # (D,)
        y = self.scrna[cell_id][gene_id]       # scalar

        return promoter, expr, y
        # return {
        #     "seq": torch.tensor(self.seq[idx], dtype=torch.float32),
        #     "signal": torch.tensor(self.signal[idx], dtype=torch.float32),
        #     "y": torch.tensor(self.y[idx], dtype=torch.float32).unsqueeze(0)
        # }
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, promoter_file, scrna_file):
        self.promoter = pd.read_csv(promoter_file)
        self.adata = sc.read(scrna_file)

        self.P = self.promoter.shape[0]
        self.C = self.adata.n_obs

        # gene_id -> index
        self.gene2idx = {
            g: i for i, g in enumerate(self.adata.var_names)
        }

    def __len__(self):
        return self.P * self.C

    def __getitem__(self, idx):
        promoter_idx = idx // self.C
        cell_idx = idx % self.C

        row = self.promoter.iloc[promoter_idx]
        gene_id = row["gene_id"]

        gene_idx = self.gene2idx[gene_id]

        promoter = row["sequence"]          # (400, 5)
        y = self.adata.X[cell_idx, gene_idx]

        # 示例：cell-level covariates
        expr = self.adata.obsm["X_pca"][cell_idx]

        return promoter, expr, float(y)

In [2]:
class PromoterOneHotEncoder:
    def __init__(self, length=400):
        self.length = length
        self.vocab = {
            "A": 0,
            "C": 1,
            "G": 2,
            "T": 3,
            "N": 4
        }
        self.num_channels = 5

    def __call__(self, seq: str):
        """
        seq: str, length <= self.length
        return: FloatTensor (length, 5)
        """
        # 初始化为 N
        one_hot = torch.zeros(self.length, self.num_channels)
        one_hot[:, 4] = 1.0  # 默认 N

        seq = seq.upper()

        for i, base in enumerate(seq[:self.length]):
            idx = self.vocab.get(base, 4)
            one_hot[i, :] = 0.0
            one_hot[i, idx] = 1.0

        return one_hot

        
class MyDataset(IterableDataset):
    def __init__(
        self,
        promoter_file,
        scrna_file,
        cell_ids_subset=None
    ):
        self.promoters = pd.read_csv(promoter_file)
        self.scrna = sc.read(scrna_file, sparse=True)

        self.promoter_encoder = PromoterOneHotEncoder(length=400)
        print("Pre-encoding promoter sequences...")
        self.promoter_tensor = self._preencode_promoters()
        print("Done.")
        #self.gene_ids = gene_ids_subset
        self.use_direct_index = cell_ids_subset is None
        self.cells = cell_ids_subset or np.arange(self.scrna.n_obs)

        # 建 promoter index -> gene_id → scrna index 的映射
        self.gene2idx = {
            g: i for i, g in enumerate(self.scrna.var.gene_id)
        }
        self.promoter2expr_idx = np.empty(len(self.promoters), dtype=np.int32)
        for i, gene_id in enumerate(self.promoters["gene_id"]):
            try:
                self.promoter2expr_idx[i] = self.gene2idx[gene_id]
            except KeyError:
                raise ValueError(f"gene_id {gene_id} not found in scRNA var")
        #print(self.gene2idx)

        self.P = len(self.promoters)
        self.C = len(self.cells)

    def _preencode_promoters(self):
        sequences = self.promoters["sequence"].values
        n = len(sequences)

        # (N, 400, 5)
        promoter_tensor = torch.zeros(
            n, 400, 5, dtype=torch.float32
        )

        for i, seq in enumerate(sequences):
            promoter_tensor[i] = self.promoter_encoder(seq)

            if i % 1000 == 0:
                print(f"  encoded {i}/{n}")

        return promoter_tensor
    
    def __len__(self):
        return self.P * self.C

    def in_getitem(self, pro_i, cell_i):
        promoter = self.promoter_tensor[pro_i]
        #gene_id = (self.promoters["gene_id"]).iloc[pro_i]
        if self.use_direct_index:
            expr_all = self.scrna[cell_i].X
        else:
            cell_id = self.cells[cell_i]
            expr_all = self.scrna[cell_id].X

        if hasattr(expr_all, "toarray"):
            expr_all = expr_all.toarray().astype("float32").squeeze()     # (16300,)
        expr_all = torch.from_numpy(expr_all).float()
        
        target_idx = self.promoter2expr_idx[pro_i]
        y = expr_all[target_idx]
        #expr_all[target_idx] = 0.0 # mask the promoter expression
        
        return promoter, expr_all, y

    def __getitem__(self, idx):
        pro_i = idx // self.C
        cell_i = idx % self.C
        
        return self.in_getitem(pro_i,cell_i)

    def __iter__(self):
        while True:
            pro_i = np.random.randint(len(self.promoters))
            cell_i = np.random.randint(len(self.cells))
            yield self.in_getitem(pro_i,cell_i)

train_dataset = MyDataset(
    promoter_file="promoter_train.csv",
    scrna_file="integrated_data.h5ad",
)
val_dataset = MyDataset(
    promoter_file="promoter_val.csv",
    scrna_file="integrated_data.h5ad",
)

Pre-encoding promoter sequences...
  encoded 0/16308
  encoded 1000/16308
  encoded 2000/16308
  encoded 3000/16308
  encoded 4000/16308
  encoded 5000/16308
  encoded 6000/16308
  encoded 7000/16308
  encoded 8000/16308
  encoded 9000/16308
  encoded 10000/16308
  encoded 11000/16308
  encoded 12000/16308
  encoded 13000/16308
  encoded 14000/16308
  encoded 15000/16308
  encoded 16000/16308
Done.
Pre-encoding promoter sequences...
  encoded 0/2794
  encoded 1000/2794
  encoded 2000/2794
Done.


In [ ]:
# dry run, valid the data tube

sample = train_dataset[0]
print(type(sample), len(sample))
promoter, expr, y = sample
print(type(promoter))
print(promoter.shape)
print(expr.shape)
print(type(y), y)

batch = next(iter(train_loader))
print(len(batch))
promoters, exprs, ys = batch
print(promoters.shape)       
print(exprs.shape)         
print(ys.shape)            

for i in range(10):
    batch = next(iter(train_loader))
    promoters, exprs, ys = batch
    print(ys)

In [3]:
# model
class SimpleGeneModel(nn.Module):
    def __init__(self, promoter_len=400, promoter_channels=5, 
                 lstm_hidden=32, expr_dim=16262):
        super().__init__()
        # promoter LSTM
        self.lstm = nn.LSTM(input_size=promoter_channels, hidden_size=lstm_hidden,
                            batch_first=True)
        # expression MLP
        self.expr_fc = nn.Linear(expr_dim, lstm_hidden)
        # output
        self.fc_out = nn.Linear(2 * lstm_hidden, 1)
        self.relu = nn.ReLU()

    def forward(self, promoter, expr):
        # promoter: (batch, 400, 5)
        # expr: (batch, 16300)
        lstm_out, _ = self.lstm(promoter)        # (batch, 400, hidden)
        lstm_out = lstm_out[:, -1, :]            # take last time step -> (batch, hidden)
        expr_out = self.relu(self.expr_fc(expr)) # (batch, hidden)
        combined = torch.cat([lstm_out, expr_out], dim=1)  # (batch, 2*hidden)
        out = self.fc_out(combined)              # (batch, 1)
        return out

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=128)

model = SimpleGeneModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model.train()

# batch = next(iter(train_loader))
# promoters, exprs, ys = batch
# print(promoters.shape, exprs.shape, ys.shape)
# ys = ys.float()

# 这里 dry run CPU
#promoters = torch.stack(promoters)  # (batch, 400, 5)
#exprs = torch.stack(exprs)          # (batch, 16300)
#ys = torch.tensor(ys).float()       # (batch,)

# 记录一个 LSTM 参数训练前的值
params_before = {
    name: p.detach().clone()
    for name, p in model.named_parameters()
}

losses = [];steps = 50
for step in range(steps):
    optimizer.zero_grad()

    batch = next(iter(train_loader))
    promoters, exprs, ys = batch
    ys = ys.float()

    out = model(promoters, exprs).squeeze(1)
    loss = criterion(out, ys)

    loss.backward()
    optimizer.step()

    losses.append(loss.item())

# 训练后的参数
for name, p in model.named_parameters():
    diff = torch.norm(p.detach() - params_before[name]).item()
    print(f"{name:30s} param_diff = {diff:.6e}")

# for name, p in model.named_parameters():
#     print(name, p.grad is None, p.grad.norm().item())

# loss 曲线
plt.figure();plt.plot(losses);plt.xlabel("Step");plt.ylabel("Loss");plt.title("Dry-run loss (same batch)");plt.show()

# 参数变化量
param_diff = (after - before).norm().item()
print("LSTM parameter change (L2 norm):", param_diff)

In [ ]:
# test whether promoters attend training ?
steps = 50              # 足够看到趋势
batch = next(iter(train_loader))
promoters, exprs, ys = batch
ys = ys.float()
model = SimpleGeneModel()
criterion = nn.MSELoss()
train_loader = DataLoader(train_dataset, batch_size=128)

def run(promoters, exprs):
    model_copy = copy.deepcopy(model)
    opt = torch.optim.Adam(model_copy.parameters(), lr=1e-4)

    losses = []
    for _ in range(steps):
        opt.zero_grad()
        out = model_copy(promoters, exprs).squeeze()
        loss = criterion(out, ys)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses
loss_A = run(promoters, exprs)
loss_B = run(torch.zeros_like(promoters), exprs)
perm = torch.randperm(promoters.size(0))
loss_C = run(promoters[perm], exprs)
loss_D = run(promoters, torch.zeros_like(exprs))

plt.figure()
plt.plot(loss_A, label="full")
plt.plot(loss_B, label="no promoter")
plt.plot(loss_C, label="shuffled promoter")
plt.plot(loss_D, label="no expr")
plt.legend()
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Promoter ablation study (single batch)")
plt.show()

In [4]:
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = SimpleGeneModel()
model = model.to(device)
print("model on GPU ready")
train_loader = DataLoader(
    train_dataset,
    batch_size=1024,          # 尽量大
    num_workers=0,           # 视 CPU 情况
    pin_memory=True,         # GPU 必开
)
val_loader = DataLoader(
    val_dataset,
    batch_size=128,          # 尽量大
    num_workers=0,           # 视 CPU 情况
    pin_memory=True,         # GPU 必开
)
print("DataLoader ready")
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
)

Using device: cuda
model on GPU ready
DataLoader ready


In [6]:
probe_batch = next(iter(train_loader))
probe_promoters, probe_exprs, probe_ys = probe_batch
probe_promoters = probe_promoters.to(device)
probe_exprs = probe_exprs.to(device)
probe_ys = probe_ys.to(device).float()
probe_steps = []
probe_loss_full = []
probe_loss_nop = []
probe_gap = []
def zero_promoter(promoters):
    return torch.zeros_like(promoters)
def ab_probe(model, promoters, exprs, ys, criterion):
    model.eval()

    # A: full
    out_full = model(promoters, exprs).squeeze(1)
    loss_full = criterion(out_full, ys).item()

    # B: no promoter
    promoters_zero = zero_promoter(promoters)
    out_nop = model(promoters_zero, exprs).squeeze(1)
    loss_nop = criterion(out_nop, ys).item()

    model.train()
    return loss_full, loss_nop

def evaluate(model, loader, mode, criterion, device, max_batches=100):
    """
    mode: "full" | "no_promoter" | "promoter_only"
    """
    model.eval() # transfer to validation mode, open dropout and set batch norm
    losses = []

    with torch.no_grad(): # turn off grad compute
        for i, (promoters, exprs, ys) in enumerate(loader):
            promoters = promoters.to(device)
            exprs = exprs.to(device)
            ys = ys.to(device).float()
    
            if mode == "no_promoter":
                promoters = zero_promoter(promoters)
            elif mode == "promoter_only":
                exprs = torch.zeros_like(exprs)
    
            out = model(promoters, exprs).squeeze(1)
            loss = criterion(out, ys)
            losses.append(loss.item())
    
            if i >= max_batches:
                break

    model.train()
    return sum(losses) / len(losses)

In [11]:
# train on gpu
model.train()
print("model train ready")
    
for step, batch in enumerate(train_loader):
    #print(step)
    promoters, exprs, ys = batch

    promoters = promoters.to(device, non_blocking=True)
    exprs = exprs.to(device, non_blocking=True)
    ys = ys.to(device, non_blocking=True).float()
    for i in range(60):
        optimizer.zero_grad()
    
        out = model(promoters, exprs).squeeze(1)
        loss = criterion(out, ys)
    
        loss.backward()
        optimizer.step()

    if step % 100 == 0:
        print(f"step {step}, loss = {loss.item():.4f}")
        
    if step % 500 == 0:
        for name, p in model.named_parameters():
            if "lstm" in name:
                print(name, p.grad.norm().item())
        loss_A, loss_B = ab_probe(
            model,
            probe_promoters,
            probe_exprs,
            probe_ys,
            criterion
        )
    
        probe_steps.append(step)
        probe_loss_full.append(loss_A)
        probe_loss_nop.append(loss_B)
        probe_gap.append(loss_B - loss_A)
    
        print(
            f"[A/B] step {step} | "
            f"full={loss_A:.4f}, noP={loss_B:.4f}, gap={loss_B-loss_A:.4f}"
        )
    if step % 2000 == 0:
        val_full = evaluate(
            model, val_loader, "full", criterion, device
        )
        val_no_p = evaluate(
            model, val_loader, "no_promoter", criterion, device
        )
        val_p_only = evaluate(
            model, val_loader, "promoter_only", criterion, device
        )
    
        print(
            f"[VAL] step {step} | "
            f"full: {val_full:.4f}, "
            f"noP: {val_no_p:.4f}, "
            f"P-only: {val_p_only:.4f}"
        )

    if step >= 4000:
        break

plt.figure(figsize=(8, 5))

plt.plot(probe_steps, probe_loss_full, label="Full (promoter + expr)")
plt.plot(probe_steps, probe_loss_nop, label="No promoter")

plt.xlabel("Training step")
plt.ylabel("Probe loss")
plt.title("A/B probe loss during training")
plt.legend()
plt.grid(True)

plt.show()

plt.figure(figsize=(8, 4))

plt.plot(probe_steps, probe_gap, marker="o")

plt.axhline(0, linestyle="--", color="gray")
plt.xlabel("Training step")
plt.ylabel("Loss gap (no-promoter − full)")
plt.title("Promoter contribution gap")
plt.grid(True)

plt.show()


Using device: cuda
model ready
DataLoader ready
model train ready
dataloader iterator ready
step 0, loss = 0.0005
lstm.weight_ih_l0 0.00010018543980550021
lstm.weight_hh_l0 4.7939633077476174e-05
lstm.bias_ih_l0 2.2124160750536248e-05
lstm.bias_hh_l0 2.2124160750536248e-05
[A/B] step 0 | full=0.0967, noP=0.0969, gap=0.0003
[VAL] step 0 | full: 0.0900, noP: 0.0905, P-only: 0.0814
step 100, loss = 0.0652
step 200, loss = 0.0573
step 300, loss = 0.0501
step 400, loss = 0.0457
step 500, loss = 0.0694
lstm.weight_ih_l0 0.011157756671309471
lstm.weight_hh_l0 0.0173169132322073
lstm.bias_ih_l0 0.01052260771393776
lstm.bias_hh_l0 0.01052260771393776
[A/B] step 500 | full=0.0863, noP=0.1866, gap=0.1003
step 600, loss = 0.0814
step 700, loss = 0.0444
step 800, loss = 0.0763
step 900, loss = 0.0650
step 1000, loss = 0.0878
lstm.weight_ih_l0 0.017584381625056267
lstm.weight_hh_l0 0.013674084097146988
lstm.bias_ih_l0 0.006859912071377039
lstm.bias_hh_l0 0.006859912071377039
[A/B] step 1000 | full=0

KeyboardInterrupt: 

In [6]:
torch.cuda.empty_cache()
GPU_BUFFER_SIZE = 8096
BATCH_SIZE = 128
steps = int(GPU_BUFFER_SIZE/BATCH_SIZE*30)
epoches = int(9e9 / GPU_BUFFER_SIZE)
print(f"epoches: {epoches}   steps: {steps}")
train_loss = [];validation_loss = []
train_loader = DataLoader(
    train_dataset,
    batch_size=GPU_BUFFER_SIZE,
    #shuffle=True,
    num_workers=0,   # Windows 必须先用 0
    pin_memory=True,              # 非常重要
    #persistent_workers=True
)
print("loader ready")
val_loader = DataLoader(
    val_dataset,
    batch_size=GPU_BUFFER_SIZE,
    #shuffle=True,
    num_workers=0,   # Windows 必须先用 0
    pin_memory=True,              # 非常重要
    #persistent_workers=True
)
model.train()
def fill_gpu_buffer(loader, device, n=GPU_BUFFER_SIZE):
    # promoters_buf = []
    # exprs_buf = []
    # ys_buf = []

    # it = iter(dataset)
    # for _ in range(n):
    #     p, e, y = next(it)
    #     promoters_buf.append(p)
    #     exprs_buf.append(torch.from_numpy(e))
    #     ys_buf.append(y)

    # promoters_buf = torch.stack(promoters_buf).to(device)
    # exprs_buf = torch.stack(exprs_buf).to(device)
    # ys_buf = torch.tensor(ys_buf, dtype=torch.float32, device=device)
    # return promoters_buf, exprs_buf, ys_buf
    p, e, y = next(iter(loader))  # 已经是 batch 了

    return (
        p.to(device, non_blocking=True),
        e.to(device, non_blocking=True),
        y.to(device, non_blocking=True),
        #torch.tensor(y, dtype=torch.float32, device=device),
    )
def gpu_batch(promoters_buf, exprs_buf, ys_buf, batch_size):
    idx = torch.randint(
        0,
        promoters_buf.size(0),
        (batch_size,),
        device=promoters_buf.device
    )

    return (
        promoters_buf[idx],
        exprs_buf[idx],
        ys_buf[idx],
    )
#prom_buf, expr_buf, y_buf = fill_gpu_buffer(train_loader, device)
for epoch in range(epoches):
    t0 = time.perf_counter()
    prom_buf, expr_buf, y_buf = fill_gpu_buffer(train_loader, device)
    #next_prom_buf, next_expr_buf, next_y_buf = fill_gpu_buffer(train_loader, device)
    t1 = time.perf_counter()
    print(f"buffer time: {t1-t0:.3f}s")
    print(f"load time: {epoches*(t1-t0)/3600} hours")
    #torch.cuda.synchronize()
    for step in range(steps):
        promoters, exprs, ys = gpu_batch(
            prom_buf, expr_buf, y_buf, BATCH_SIZE
        )
    
        optimizer.zero_grad()
        out = model(promoters, exprs).squeeze()
        loss = criterion(out, ys)
        loss.backward()
        optimizer.step()
    #torch.cuda.synchronize()
    t2 = time.perf_counter()
    print(f"train time: {t2-t1:.3f}s")
    #prom_buf, expr_buf, y_buf = next_prom_buf, next_expr_buf, next_y_buf
    if epoch % 2 == 0:
        print(loss.item())
        train_loss.append(loss.item())
        loss_A, loss_B = ab_probe(
            model,
            probe_promoters,
            probe_exprs,
            probe_ys,
            criterion
        )
    
        probe_steps.append(epoch/30000)
        probe_loss_full.append(loss_A)
        probe_loss_nop.append(loss_B)
        probe_gap.append(loss_B - loss_A)
    
        print(
            f"[A/B] step {step} | "
            f"full={loss_A:.4f}, noP={loss_B:.4f}, gap={loss_B-loss_A:.4f}"
        )

        # val_full = evaluate(
        #     model, val_loader, "full", criterion, device
        # )
        # val_no_p = evaluate(
        #     model, val_loader, "no_promoter", criterion, device
        # )
        # val_p_only = evaluate(
        #     model, val_loader, "promoter_only", criterion, device
        # )
        # validation_loss.append(val_full)
        # print(
        #     f"[VAL] step {step} | "
        #     f"full: {val_full:.4f}, "
        #     f"noP: {val_no_p:.4f}, "
        #     f"P-only: {val_p_only:.4f}"
        # )
    t3 = time.perf_counter()
    print(f"validataion time: {t3-t2:.3f}s")

plt.figure(figsize=(8, 5))
plt.plot(probe_steps, probe_loss_full, label="Full (promoter + expr)")
plt.plot(probe_steps, probe_loss_nop, label="No promoter")
plt.xlabel("Training step")
plt.ylabel("Probe loss")
plt.title("A/B probe loss during training")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(probe_steps, probe_gap, marker="o")
plt.axhline(0, linestyle="--", color="gray")
plt.xlabel("Training step")
plt.ylabel("Loss gap (no-promoter − full)")
plt.title("Promoter contribution gap")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(probe_steps, train_loss, label="train loss")
plt.plot(probe_steps, validation_loss, label="validation loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

epoches: 1111660   steps: 1897
loader ready
buffer time: 22.640s
load time: 6991.092449997291 hours
train time: 9.646s
0.00025493279099464417


NameError: name 'ab_probe' is not defined